In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np

In [26]:
# =========================================================
# USER INPUT FILES
# =========================================================
LINE_FILE = r"F:\P-52797\PS 694 Companiganj Noakhali\GeoJasionFile\ThalweghLine.geojson"
SURVEY_FILE = r"F:\P-52797\PS 694 Companiganj Noakhali\GeoJasionFile\XS_from_Nak_Geojason.geojson"
OUTPUT_FILE = r"F:\P-52797\PS 694 Companiganj Noakhali\GeoJasionFile\Thalweg_3m_Points_RL.geojson"

# =========================================================
# 1. LOAD DATA
# =========================================================
line_gdf = gpd.read_file(LINE_FILE)
survey_gdf = gpd.read_file(SURVEY_FILE)

# Ensure same CRS (must be projected in meters)
survey_gdf = survey_gdf.to_crs(line_gdf.crs)

In [27]:
line_gdf.head(10)

,id,geometry
0,1,"MULTILINESTRING ((329938.261 2535991.391, 3296..."
1,2,"MULTILINESTRING ((329932.351 2536006.28, 32976..."
2,3,"MULTILINESTRING ((329950 2535964, 329797 25358..."


In [28]:
survey_gdf.head(10)

,Chainage,X,Y,Distance_m,RL_mMSL,Remarks,geometry
0,17240.0,330099.053,2531693.153,0.0,2.534,None,POINT (330099.053 2531693.153)
1,17240.0,330095.718,2531696.106,4.0,2.645,None,POINT (330095.718 2531696.106)
2,17240.0,330094.526,2531698.458,7.0,2.731,None,POINT (330094.526 2531698.458)
3,17240.0,330093.971,2531702.576,11.0,2.679,None,POINT (330093.971 2531702.576)
4,17240.0,330093.838,2531704.391,12.0,2.698,None,POINT (330093.838 2531704.391)
5,17240.0,330091.303,2531706.929,16.0,2.639,None,POINT (330091.303 2531706.929)
6,17240.0,330089.193,2531709.086,19.0,2.464,None,POINT (330089.193 2531709.086)
7,17240.0,330087.369,2531710.838,21.0,2.395,None,POINT (330087.369 2531710.838)
8,17240.0,330085.950,2531712.982,24.0,2.424,None,POINT (330085.95 2531712.982)
9,17240.0,330083.956,2531715.124,27.0,2.514,None,POINT (330083.956 2531715.124)


In [29]:
INTERVAL = 3        # meters
BUFFER_DIST = 3     # meters

# =========================================================
# 2. FUNCTION: CREATE POINTS ALONG LINE
# =========================================================
def create_points_along_line(line, interval):
    distances = np.arange(0, line.length + interval, interval)
    points = [line.interpolate(d) for d in distances]
    return points, distances

# =========================================================
# 3. GENERATE POINTS FOR EACH LINE FEATURE
# =========================================================
rows = []

for _, row in line_gdf.iterrows():
    line_id = row["id"]  # change if field name differs
    points, dists = create_points_along_line(row.geometry, INTERVAL)

    for pt, dist in zip(points, dists):
        rows.append({
            "line_id": line_id,
            "Distance": dist,
            "geometry": pt
        })

points_gdf = gpd.GeoDataFrame(rows, crs=line_gdf.crs)

# =========================================================
# 4. EASTING / NORTHING
# =========================================================
points_gdf["Easting"] = points_gdf.geometry.x
points_gdf["Northing"] = points_gdf.geometry.y

points_gdf.head(20)

,line_id,Distance,geometry,Easting,Northing
0,1,0.0,POINT (329938.261 2535991.391),329938.260900,2.535991e+06
1,1,3.0,POINT (329935.883 2535989.561),329935.883347,2.535990e+06
2,1,6.0,POINT (329933.506 2535987.732),329933.505794,2.535988e+06
3,1,9.0,POINT (329931.128 2535985.902),329931.128242,2.535986e+06
4,1,12.0,POINT (329928.751 2535984.073),329928.750689,2.535984e+06
5,1,15.0,POINT (329926.373 2535982.243),329926.373136,2.535982e+06
6,1,18.0,POINT (329923.996 2535980.414),329923.995583,2.535980e+06
7,1,21.0,POINT (329921.618 2535978.584),329921.618031,2.535979e+06
8,1,24.0,POINT (329919.24 2535976.755),329919.240478,2.535977e+06
9,1,27.0,POINT (329916.863 2535974.925),329916.862925,2.535975e+06


In [30]:
# =========================================================
# 5. BUFFER + SPATIAL JOIN
# =========================================================
buffered = points_gdf.copy()
buffered["geometry"] = buffered.geometry.buffer(BUFFER_DIST)

joined = gpd.sjoin(
    buffered,
    survey_gdf[["RL_mMSL", "geometry"]],
    how="left",
    predicate="intersects"
)

# =========================================================
# 6. SELECT NEAREST SURVEY POINT RL (FINAL & CORRECT)
# =========================================================

def compute_dist(row):
    if pd.isna(row["index_right"]):
        return np.nan
    return row.geometry.centroid.distance(
        survey_gdf.loc[row["index_right"]].geometry
    )

# Compute distance safely
joined["dist"] = joined.apply(compute_dist, axis=1)

# Sort by distance (nearest first)
joined = joined.sort_values("dist")

# IMPORTANT: group by ORIGINAL thalweg point index
nearest_rl = (
    joined
    .groupby(joined.index)
    .first()["RL_mMSL"]
)

# Assign back
points_gdf["RLmMSL"] = nearest_rl

points_gdf.head(50)

,line_id,Distance,geometry,Easting,Northing,RLmMSL
0,1,0.0,POINT (329938.261 2535991.391),329938.260900,2.535991e+06,-1.63
1,1,3.0,POINT (329935.883 2535989.561),329935.883347,2.535990e+06,NaN
2,1,6.0,POINT (329933.506 2535987.732),329933.505794,2.535988e+06,NaN
3,1,9.0,POINT (329931.128 2535985.902),329931.128242,2.535986e+06,NaN
4,1,12.0,POINT (329928.751 2535984.073),329928.750689,2.535984e+06,NaN
5,1,15.0,POINT (329926.373 2535982.243),329926.373136,2.535982e+06,NaN
6,1,18.0,POINT (329923.996 2535980.414),329923.995583,2.535980e+06,NaN
7,1,21.0,POINT (329921.618 2535978.584),329921.618031,2.535979e+06,NaN
8,1,24.0,POINT (329919.24 2535976.755),329919.240478,2.535977e+06,NaN
9,1,27.0,POINT (329916.863 2535974.925),329916.862925,2.535975e+06,NaN


In [31]:
# =========================================================
# 7. CONTROLLED INTERPOLATION (NO EXTRAPOLATION)
# =========================================================
points_gdf.sort_values(["line_id", "Distance"], inplace=True)

points_gdf["RLmMSL"] = (
    points_gdf
    .groupby("line_id")["RLmMSL"]
    .transform(lambda s: s.interpolate(method="linear", limit_area="inside"))
)

print("✅ PROCESS COMPLETED SUCCESSFULLY")

points_gdf.head(20)

✅ PROCESS COMPLETED SUCCESSFULLY


,line_id,Distance,geometry,Easting,Northing,RLmMSL
0,1,0.0,POINT (329938.261 2535991.391),329938.260900,2.535991e+06,-1.630000
1,1,3.0,POINT (329935.883 2535989.561),329935.883347,2.535990e+06,-1.626970
2,1,6.0,POINT (329933.506 2535987.732),329933.505794,2.535988e+06,-1.623939
3,1,9.0,POINT (329931.128 2535985.902),329931.128242,2.535986e+06,-1.620909
4,1,12.0,POINT (329928.751 2535984.073),329928.750689,2.535984e+06,-1.617879
5,1,15.0,POINT (329926.373 2535982.243),329926.373136,2.535982e+06,-1.614848
6,1,18.0,POINT (329923.996 2535980.414),329923.995583,2.535980e+06,-1.611818
7,1,21.0,POINT (329921.618 2535978.584),329921.618031,2.535979e+06,-1.608788
8,1,24.0,POINT (329919.24 2535976.755),329919.240478,2.535977e+06,-1.605758
9,1,27.0,POINT (329916.863 2535974.925),329916.862925,2.535975e+06,-1.602727


In [32]:
# =========================================================
# 8. SAVE OUTPUT
# =========================================================
points_gdf.to_file(OUTPUT_FILE, driver="GeoJSON")